# Notebook 02: Concurrencia, Asincronía y asyncio

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sonder-art/fdd_p26/blob/main/clase/16_computo/code/02_concurrencia_asyncio.ipynb)

**Módulo 16 — Clase 2**

Este notebook acompaña los archivos `03_concurrencia_y_asincronia.md` y `04a_asyncio_fundamentos.md`.

Secciones **** se trabajan durante la sesión.  
Secciones **** se completan después.

---

In [ ]:
import asyncio
import time
import threading
import os
import sys

print(f'Python {sys.version}')
print(f'asyncio version: {asyncio.__version__ if hasattr(asyncio, "__version__") else "built-in"}')

# Jupyter ya tiene un event loop corriendo — podemos usar await directamente en las celdas
# Si usas un script .py, necesitas asyncio.run(main())

## Sección 1: await secuencial vs asyncio.gather — la diferencia central

**Qué vamos a ver:** la diferencia entre M2 (await secuencial) y M4 (gather) es literalmente una línea de código. Los tiempos medidos hacen que la diferencia sea imposible de ignorar.

**Predicción del modelo:**
- M2 (await en secuencia): `T_total = N × T_tarea` — cada usuario espera a que el anterior termine
- M4 (gather): `T_total ≈ T_tarea_más_lenta` — todos los usuarios esperan *al mismo tiempo*

Con N=5 tareas de 1.0s cada una:
- M2 debería tardar: **5.0s**
- M4 debería tardar: **~1.0s**

Corre las celdas y verifica. ¿Coincide con la predicción? ¿El speedup es exactamente 5×, o hay overhead?

> Referencia: `04a_asyncio_fundamentos.md` — sección "asyncio.gather — M4 en una línea"

In [ ]:
# Tarea simulada con I/O-bound: espera τ segundos
async def tarea_io(nombre: str, duracion: float) -> str:
    # exec(τᵢ): inicializar
    inicio = time.perf_counter()
    # wait(τᵢ): simula I/O (llamada a API, lectura de BD, etc.)
    await asyncio.sleep(duracion)
    # exec(τᵢ): procesar resultado
    elapsed = time.perf_counter() - inicio
    return f'{nombre}: {elapsed:.2f}s'

DURACION = 1.0  # cada tarea tarda 1s de I/O
N_TAREAS = 5

# --- M2: await secuencial (esperas NO explotadas) ---
t0 = time.perf_counter()
resultados_m2 = []
for i in range(N_TAREAS):
    r = await tarea_io(f'τ{i+1}', DURACION)
    resultados_m2.append(r)
t_m2 = time.perf_counter() - t0

print(f'=== M2: await secuencial ===')
for r in resultados_m2:
    print(f'  {r}')
print(f'Tiempo total M2: {t_m2:.2f}s  (esperado: {N_TAREAS * DURACION:.1f}s = N×T)')
print()

In [ ]:
# --- M4: asyncio.gather (esperas SÍ explotadas) ---
t0 = time.perf_counter()
resultados_m4 = await asyncio.gather(
    *[tarea_io(f'τ{i+1}', DURACION) for i in range(N_TAREAS)]
)
t_m4 = time.perf_counter() - t0

print(f'=== M4: asyncio.gather ===')
for r in resultados_m4:
    print(f'  {r}')
print(f'Tiempo total M4: {t_m4:.2f}s  (esperado: ~{DURACION:.1f}s = T_max)')
print()
print(f'Speedup M4/M2: {t_m2/t_m4:.1f}x')
print()
print(f'Conclusión: gather explota las esperas — exec(τⱼ) ∩ wait(τᵢ) ≠ ∅')
print(f'Las {N_TAREAS} tareas de {DURACION}s corren en ~{DURACION}s en lugar de {N_TAREAS*DURACION}s')

## Sección 2: Traza del event loop con asyncio debug mode

**Qué vamos a ver:** asyncio tiene un modo de depuración que emite advertencias cuando el event loop se bloquea más tiempo del esperado. Es la herramienta para diagnosticar el anti-patrón de `time.sleep` dentro de funciones `async`.

**El problema a demostrar:**
```
time.sleep(0.3)       ← bloquea el hilo del OS durante 300ms
                         → el event loop no puede ejecutar NINGUNA otra coroutine
                         → gather con 3 tareas de 0.3s tarda 0.9s (secuencial)

await asyncio.sleep(0.3)  ← registra un callback y cede el control
                              → el event loop puede ejecutar otras coroutines
                              → gather con 3 tareas de 0.3s tarda ~0.3s (M4)
```

**Predicción:**
- `gather` con `asyncio.sleep(0.3)`: debería tardar **~0.3s**
- `gather` con `time.sleep(0.3)`: debería tardar **~0.9s** (sin mejora — M1)

El modo debug (`loop.set_debug(True)`) detectará el bloqueo y emitirá una advertencia en el segundo caso. Observa el output completo.

> Referencia: `04a_asyncio_fundamentos.md` — sección "time.sleep vs asyncio.sleep"

In [ ]:
import asyncio
import time

# Habilitamos debug mode para ver bloqueos
loop = asyncio.get_event_loop()
loop.set_debug(True)

# Un umbral bajo para detectar bloqueos rápidamente
# (normalmente el umbral es 100ms)
loop.slow_callback_duration = 0.05  # 50ms

# Tarea bien escrita: libera el event loop
async def tarea_correcta(nombre: str):
    print(f'  {nombre}: inicio')
    await asyncio.sleep(0.3)   # wait(τ) — event loop libre
    print(f'  {nombre}: fin')

# Tarea mal escrita: BLOQUEA el event loop
async def tarea_bloqueante(nombre: str):
    print(f'  {nombre}: inicio')
    time.sleep(0.3)            # ← bloquea el hilo del OS entero
    print(f'  {nombre}: fin')

# ¿Qué diferencia ves en la salida?
print('=== gather con tareas CORRECTAS (asyncio.sleep) ===')
t0 = time.perf_counter()
await asyncio.gather(tarea_correcta('A'), tarea_correcta('B'), tarea_correcta('C'))
print(f'Tiempo: {time.perf_counter()-t0:.2f}s  (esperado: ~0.3s)\n')

print('=== gather con tareas BLOQUEANTES (time.sleep) ===')
t0 = time.perf_counter()
await asyncio.gather(tarea_bloqueante('X'), tarea_bloqueante('Y'), tarea_bloqueante('Z'))
print(f'Tiempo: {time.perf_counter()-t0:.2f}s  (esperado: ~0.9s — sin mejora)')
print()
print('Observa: con time.sleep, gather NO ayuda.')
print('time.sleep bloquea el event loop → ninguna otra coroutine puede avanzar.')

In [ ]:
# Desactivar debug mode para el resto del notebook
loop.set_debug(False)

---

## Sección 3: Implementar M2 y M3 — por qué NO mejoran

**TAREA — implementación guiada**

Esta sección te pide implementar dos modelos *incorrectos* para CPU-bound e ineficientes para sus casos de uso, y medir por qué fallan la promesa de concurrencia/paralelismo.

**TAREA 3.1 — M2: async con await secuencial**

Ya viste M2 en la Sección 1. Ahora explícalo formalmente:
- ¿Qué condición de M4 (`exec(τⱼ) ∩ wait(τᵢ) ≠ ∅`) falla en M2?
- ¿En qué se diferencia `await fn1(); await fn2()` de `asyncio.gather(fn1(), fn2())`?

Escribe la respuesta como comentario en la celda antes de correr el código.

**TAREA 3.2 — M3: threading CPU-bound**

El GIL (Global Interpreter Lock) garantiza que solo un hilo ejecuta bytecode Python a la vez. Para tareas CPU-bound (`wait(τᵢ) = ∅`), el GIL nunca se libera — el threading no puede producir paralelismo real.

**Predicción:** con N=4 hilos en una tarea CPU-bound, el speedup esperado es **≈1×** (sin mejora). Puede incluso ser < 1× por el overhead de sincronización del GIL.

Implementa el threading y mide. ¿Coincide con la predicción?

In [ ]:
# TAREA 3.1 — M2: async con await secuencial (ya visto en Sección 1)
# Pregunta: ¿por qué M2 es idéntico a M1 en términos de tiempo?
# Responde con la definición formal: ¿qué condición de M4 falta en M2?

# TAREA 3.2 — M3: threading CPU-bound
# Implementa N tareas CPU-bound con threading y mide vs secuencial.
# ¿Coincide con la predicción del GIL (sin speedup, posible slowdown)?

def tarea_cpu_bound(n: int) -> int:
    """Tarea CPU-bound pura: wait(τᵢ) = ∅"""
    return sum(range(n))

N_CPU = 30_000_000
N_HILOS = 4

# --- Secuencial (M1) ---
t0 = time.perf_counter()
for _ in range(N_HILOS):
    tarea_cpu_bound(N_CPU)
t_secuencial = time.perf_counter() - t0

# --- Threading M3 ---
# TODO: implementa con threading.Thread y mide el tiempo
# Pista: usa la misma tarea_cpu_bound con N_HILOS hilos

# TODO: imprime los tiempos y el speedup
# ¿Qué dice el resultado sobre M3 + GIL en Python?

print(f'M1 secuencial: {t_secuencial:.2f}s')
print('M3 threading: TODO — implementa aquí')

## Sección 4: Race condition reproducible + fix con Lock

**TAREA — reproducir y corregir**

Las condiciones de carrera (race conditions) son el bug clásico de la concurrencia con memoria compartida. Ocurren cuando múltiples hilos leen y escriben la misma variable sin coordinación.

**Por qué ocurre:**
La operación `contador += 1` parece atómica pero en realidad son 3 pasos:
```
LOAD  contador      → lee el valor actual al registro de la CPU
ADD   registro, 1   → incrementa en 1
STORE registro → contador  → escribe de vuelta
```
Si dos hilos ejecutan esto concurrentemente, el segundo puede leer el valor *antes* de que el primero escriba su resultado — se pierde un incremento.

**Predicción:**
- Sin lock: `contador` final < `N_INCREMENTOS × N_HILOS` (incrementos perdidos, no determinístico)
- Con `threading.Lock`: `contador` final = `N_INCREMENTOS × N_HILOS` siempre

Corre varias veces sin lock. ¿El resultado es siempre distinto? ¿Siempre menor que el esperado?

> Nota: el GIL de Python reduce (no elimina) las race conditions. Este ejemplo las reproduce porque el increment no es atómico incluso con el GIL.

In [ ]:
import threading

# TAREA 4.1 — Reproduce la race condition
N_INCREMENTOS = 100_000
N_HILOS_RACE = 4

# Sin lock — resultado no determinista
contador_sin_lock = [0]

def incrementar_sin_lock():
    for _ in range(N_INCREMENTOS):
        contador_sin_lock[0] += 1   # NO atómico: LOAD, ADD, STORE separados

hilos = [threading.Thread(target=incrementar_sin_lock) for _ in range(N_HILOS_RACE)]
for h in hilos: h.start()
for h in hilos: h.join()

esperado = N_INCREMENTOS * N_HILOS_RACE
print(f'Sin lock  — esperado: {esperado:,}, obtenido: {contador_sin_lock[0]:,}')
print(f'Diferencia: {esperado - contador_sin_lock[0]:,} incrementos perdidos')
print()

# TAREA 4.2 — Fix con Lock
# TODO: implementa incrementar_con_lock usando threading.Lock
# El resultado debe ser siempre exactamente N_INCREMENTOS × N_HILOS_RACE

# TODO: verifica que contador_con_lock == esperado

## Sección 5: Chatbot v2 con asyncio — N usuarios concurrentes

**TAREA — implementación completa**

Implementa el servidor chatbot v2 del Escenario A: LLM como API remota, I/O-bound. Compara la versión secuencial (v1) con la concurrente (v2).

**Arquitectura del chatbot v2:**
```
N usuarios simultáneos
        │
asyncio.gather(handle_request(0), handle_request(1), ..., handle_request(N-1))
        │
  ┌─────┴──────────────────────────────────────────┐
  │  Event loop (1 hilo)                           │
  │                                                │
  │  τ_u0 exec → await BD(50ms) → await LLM(1.5s) │
  │    τ_u1 exec → await BD(50ms) → await LLM(1.5s)│
  │      τ_u2 exec → await BD(50ms) → await LLM...│
  └──────────────────┬─────────────────────────────┘
                     │ (simultáneamente)
            [BD asyncpg]  [LLM API aiohttp]
```

**Predicciones:**
- v1 (secuencial) con N=10: `T_total ≈ 10 × 1.55s = 15.5s`
- v2 (gather) con N=10: `T_total ≈ 1.55s`
- Latencia del usuario 10 en v2: **similar a la del usuario 1** (todos esperan en paralelo)

Implementa `servidor_v1` y `servidor_v2`, mide con N=10, y responde:
1. ¿La latencia es uniforme entre usuarios en v2? ¿Por qué?
2. ¿Qué pasaría si uno de los usuarios tuviera `time.sleep` en su handler?

> Referencia: `04a_asyncio_fundamentos.md` — sección "Chatbot v2"

In [ ]:
import asyncio
import time
import random

# Operaciones I/O-bound del chatbot (simuladas)
async def consultar_bd(user_id: int) -> list:
    """wait(τᵢ): I/O a base de datos — ~50ms"""
    await asyncio.sleep(0.05)
    return [f'historial de usuario {user_id}']

async def llamar_llm(historial: list) -> str:
    """wait(τᵢ): I/O a API del LLM — 1–2s variable"""
    await asyncio.sleep(random.uniform(1.0, 2.0))
    return f'respuesta para: {historial[-1]}'

async def handle_request(user_id: int) -> dict:
    """Una petición completa del chatbot v2 (M4)"""
    # exec(τᵢ)
    t_inicio = time.perf_counter()

    # wait(τᵢ): BD
    historial = await consultar_bd(user_id)

    # wait(τᵢ): LLM
    respuesta = await llamar_llm(historial)

    # exec(τᵢ)
    latencia = time.perf_counter() - t_inicio
    return {'user': user_id, 'respuesta': respuesta, 'latencia': latencia}

# TAREA 5.1 — Servidor secuencial (chatbot v1 como baseline)
async def servidor_v1(n_usuarios: int):
    """M1: un usuario a la vez"""
    # TODO: implementa el servidor secuencial
    # Hint: un for loop con await handle_request(i) para cada usuario
    pass

# TAREA 5.2 — Servidor concurrente (chatbot v2)
async def servidor_v2(n_usuarios: int):
    """M4: todos los usuarios concurrentes con gather"""
    # TODO: implementa con asyncio.gather
    pass

# TAREA 5.3 — Compara v1 vs v2 con N=10 usuarios
# Mide tiempos totales y latencias promedio
# ¿Cuántas veces más rápido es v2?
# ¿La latencia de cada usuario es similar en v2? ¿Por qué?

N = 10
print(f'Comparando v1 vs v2 con {N} usuarios...')
print('TODO: implementa servidor_v1 y servidor_v2 arriba')